# CSA-3D: basic functionality

This notebook walks through the core CSA-3D workflow and doubles as a smoke test: every
section ends with `assert` checks, so running it top-to-bottom without errors means the
basic functionality works.

**First run:** needs internet access. `setup_data()` downloads ~400 MB of reference data
(skipped on later runs) and `Mcsa.build()` downloads the assembly structures it needs.

## 1. Set up the data

`setup_data()` downloads everything to `~/.csa3d/data` (or a path you pass) and makes it the active data directory. Files already present are skipped, so it is cheap to re-run.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

import csa3d
print("CSA-3D", csa3d.__version__)

assert csa3d.setup_data(), "data setup failed - see the log above"
print("Data directory:", csa3d.get_data_dir())
assert csa3d.validate_config()

INFO: Enabling RDKit 2026.09.1 jupyter extensions


INFO: Setting up CSA-3D data in /Users/rizioti/.csa3d/data


INFO: Already present, skipping: sifts/uniprot_pdb.csv


INFO: Already present, skipping: sifts/pdb_chain_uniprot.csv


INFO: Already present, skipping: sifts/pdb_chain_enzyme.csv


INFO: Already present, skipping: pdbe/resolu.idx


... (12 outputs left out)

INFO:   MCSA   ok


INFO:   RHEA   ok


INFO:   KEGG   ok


CSA-3D 0.2.2
Data directory: /Users/rizioti/.csa3d/data


## 2. Load the M-CSA database and build an entry

`Mcsa()` loads the catalytic residue annotations; `build()` assembles the catalytic sites for the requested entries, downloading any missing assembly structures from PDBe.

In [2]:
logging.getLogger("csa3d").setLevel(logging.WARNING)  # hide per-structure download messages

db = csa3d.Mcsa()
print(len(db.catalytic_residue_info), "M-CSA entries available")
assert len(db.catalytic_residue_info) > 900

assert db.build(entry_ids=1, annotate=True)
entry = db.entries[1]
print(f"M-CSA {entry.mcsa_id}: {entry.info['enzyme_name']} (EC {', '.join(entry.info['all_ecs'])})")
print(len(entry.pdbsites), "PDB sites,", len(entry.unisites), "UniProt sites")
assert entry.pdbsites and entry.unisites

1002 M-CSA entries available


M-CSA 1: glutamate racemase (EC 5.1.1.3)
53 PDB sites, 336 UniProt sites


## 3. Inspect a catalytic site

Each entry has one *reference* site (the structure M-CSA annotated) and homologous sites mapped onto it.

In [3]:
reference = next(site for site in entry.get_pdbsites() if site.is_reference)
print("Site:", reference.id)
print("PDB:", reference.pdb_id, "| UniProt:", reference.uniprot_id, "| EC:", reference.ec,
      "| resolution:", reference.resolution, "Å")

print("\nCatalytic residues:")
for res in reference.residues:
    print(f"  {res.resname} {res.chain}{res.auth_resid}  {', '.join(res.funclocs)}")

print("\nLigands within 5 Å:")
for het in reference.ligands:
    print(f"  {het.resname} {het.chain}{het.resid}  type={het.type}  similarity={het.similarity}")

assert reference.pdb_id and len(reference.residues) > 0
assert reference.uniprot_id is not None and reference.ec is not None  # SIFTS mappings loaded

Site: 1b73_A7-A70-A178-A180-A8-AA147


PDB: 1b73 | UniProt: P56868 | EC: 5.1.1.3 | resolution: 2.3 Å

Catalytic residues:
  Asp A7  
  Cys A70  
  Cys A178  
  His A180  
  Ser A8  
  Glu AA147  

Ligands within 5 Å:


## 4. Compare two sites

`fit()` superimposes the mobile site onto the fixed one and returns `(rotation, translation, rms, rms_all)`: `rms` excludes outlying atoms after the fitting cycles, `rms_all` includes every atom.

In [4]:
sites = list(entry.get_pdbsites(sane_only=True))
print(len(sites), "sane sites")
other = next(site for site in sites if not site.is_reference and site.is_conserved and not site.get_gaps())

rot, tran, rms, rms_all = reference.fit(other, weighted=True)
print(f"{reference.id} vs {other.id}: RMSD {rms:.2f} Å (all atoms: {rms_all:.2f} Å)")
assert rot.shape == (3, 3) and rms_all >= 0

48 sane sites
1b73_A7-A70-A178-A180-A8-AA147 vs 4b1f_B7-B70-B181-B183-B8-B150: RMSD 0.97 Å (all atoms: 3.98 Å)


## 5. Cluster sites by geometry

In [5]:
matrix = entry.rmsd_matrix(sites)
print("RMSD matrix:", matrix.shape)

clusters = entry.clustering(matrix)
print(len(clusters), "clusters")
for cluster_id, site_ids in clusters.items():
    print(f"  cluster {cluster_id}: {len(site_ids)} sites")

assert matrix.shape[0] == matrix.shape[1] > 1
assert sum(len(ids) for ids in clusters.values()) == matrix.shape[0]

RMSD matrix: (48, 48)
2 clusters
  cluster 0: 47 sites
  cluster 1: 1 sites


## 6. Build a 3D template

`create_template()` averages the conserved sites into a consensus geometry and writes it as a template for [Jess](https://github.com/iriziotis/jess) / [PyJess](https://github.com/althonos/pyjess), which search protein structures for matches to the template.

In [6]:
import tempfile
from pathlib import Path

outdir = Path(tempfile.mkdtemp(prefix="csa3d_test_"))
template = entry.create_template(outdir=str(outdir))
written = sorted(p.name for p in outdir.iterdir())
print("Template built from reference", template.id)
print("Written to", outdir, ":", written)

assert template is not None and written

Template built from reference 2jfw_A14-A77-A188-A190-A15-A156
Written to /tmp/csa3d_test_s9f2c7ha : ['csa3d_0001.all.2jfw_A14-A77-A188-A190-A15-A156.template.pdb']


## 7. Export a site as PDB

In [7]:
reference.write_pdb(outdir=str(outdir), write_hets=True)
pdb_files = sorted(outdir.glob("mcsa_*.pdb"))
print([p.name for p in pdb_files])
print(pdb_files[0].read_text().splitlines()[0])
assert pdb_files

['mcsa_0001.1b73_A7-A70-A178-A180-A8-AA147.reference.c.all.sane.pdb']
REMARK CATALYTIC SITE


## 8. Sequence-level (UniProt) sites

In [8]:
ref_unisite = next(site for site in entry.unisites if site.is_reference)
print("Reference UniProt site:", ref_unisite.id, ref_unisite.sequence)
conserved = sum(site.is_conserved for site in entry.unisites)
print(f"{conserved}/{len(entry.unisites)} UniProt sites are conserved")

entry.update_mapped_sites()
mapped = sum(bool(site.mapped_pdbsites) for site in entry.unisites)
print(mapped, "UniProt sites map to PDB sites via SIFTS")
assert ref_unisite.sequence

Reference UniProt site: P56868 DCCHSE
329/336 UniProt sites are conserved
11 UniProt sites map to PDB sites via SIFTS


## 9. Structures are cached in the data directory

In [9]:
from pathlib import Path
cif_dir = Path(csa3d.get_data_dir()) / "pdbe" / "assembly_cif" / "cif"
cifs = list(cif_dir.glob("*.cif"))
print(len(cifs), "assembly files cached in", cif_dir)
assert any(p.name.startswith(reference.pdb_id) for p in cifs)
print("All checks passed")

37 assembly files cached in /Users/rizioti/.csa3d/data/pdbe/assembly_cif/cif
All checks passed
